In [2]:
import pandas as pd
import json
import os,re
import random
from openai import OpenAI #estamos la clase concreta OpenAI del módulo openai
from dotenv import load_dotenv #importamos una función concreta del módulo
load_dotenv("template.env")

True

In [3]:
# Acceder a la clave de API de OpenAI
api_key = os.getenv("OPENAI_API_KEY")

# Asegurarte de que la clave de API se haya cargado correctamente
if api_key is None:
	raise ValueError("La clave de API no está configurada en el archivo .env")

client = OpenAI() #creando un objeto de la clase"

In [4]:
dataset_folder = os.getenv("DATASET_FOLDER")
dataset_name = "AoA_Ratings_Spanish_for_second_finetuning.xlsx"
#dataset_name = "Alonso_2014_SpanishAoA.xlsx"
dataset_path = str(dataset_folder) + dataset_name

df = pd.read_excel(dataset_path)
#df = df.sample(100)
df.head()

,word,Blup_maxreplace
0,abismado,20.387835
1,ablande,12.620635
2,absorber,9.583546
3,abstraído,15.166971
4,abulense,15.278792


In [5]:
#FUNCTION DECLARATION

def create_fine_tunning_from_json(json_object,prompt,array):
	word = json_object[array[0]]
	aoa = json_object[array[1]]
	f_t_line = generate_fine_tuning(
		prompt,word,aoa
		)
	return f_t_line

def word_into_prompt(prompt,word):
	return prompt.replace("{palabra}",word)

def word_aoa_into_answer(word,aoa):
	return json.dumps({"Word": str(word) , "AoA" : str(round(aoa,2))})

def generate_fine_tuning(prompt,word,aoa):
	f_t_line = {
		"messages": [
			{ "role": "user", "content": word_into_prompt(prompt,word) },
			{ "role": "assistant", "content": word_aoa_into_answer(word,aoa) }
		]}
	return f_t_line

def create_file_from_tasks(tasks,file_name):
	with open(file_name, 'w') as file:
		for obj in tasks:
			file.write(json.dumps(obj) + '\n')

def create_f_t_array_from_dataframe(df,prompt,array):
	tasks = []
	for index, row in df.iterrows():
		task = create_fine_tunning_from_json(row,prompt,array)
		tasks.append(task)
	return tasks


def get_line_file(file_name,line,extract_func):
	with open(file_name, 'r') as f:
		for line_number, theline in enumerate(f):
			if line_number == line:
				res = theline
				break
	res = json.loads(res)
	return extract_func(res)


def upload_file(file_name: str, purpose: str) -> str:
    with open(file_name, "rb") as file_fd:
        response = client.files.create(file=file_fd, purpose=purpose)
    return response.id

In [6]:
#PROMPTS

#AGE PROMPT
categorize_system_prompt_paraphrase = '''
La edad de adquisición (AoA) de una palabra se refiere a la edad en la que se aprendió una palabra por primera vez.
En concreto, cuándo una persona habría entendido por primera vez esa palabra si alguien la hubiera utilizado delante de ella, incluso cuando aún no la hubiera dicho, leído o escrito.
Estima la edad media de adquisición (AoA) de la palabra "{palabra}" para un hablante nativo de español.
El formato de salida debe ser un objeto JSON. Por ejemplo: { Word: {palabra} , AoA: //AoA de la palabra expresado en años, puede incluir decimales}
'''

In [7]:
#SET output folder
output_folder = os.getenv("OUTPUT_FOLDER")
def out_file(file_name): return (str(output_folder) + file_name)

middle_folder = os.getenv("MIDDLE_FOLDER")
def middle_file(file_name): return (str(middle_folder) + file_name)

In [8]:
f_t_file_array = [middle_file("batch_job_mmlu_f_t_aoa_alonso_second.jsonl"),
				  middle_file("batch_job_mmlu_check_aoa_alonso_second.jsonl"),
				  middle_file("batch_job_mmlu_batch_aoa_alonso_second.jsonl")]

In [ ]:
#CREATE tasks from the database
array = ["word","Blup_maxreplace"]
#array = ["word","averageAoA"]
f_t_array = [create_f_t_array_from_dataframe(df,categorize_system_prompt_paraphrase,array)]

#SEPARATE words in piles to fine-tune or check
num_task_train = 2000

KeyError: 'averageAoA'

In [9]:
#check all tasks in task array
max_num_task = 0
for i in range(0,len(f_t_array)):
	max_num_task += len(f_t_array[i])

#generate random task indexes
if (max_num_task>num_task_train):
	random_num_array = []
	for i in range(0,num_task_train):
		new_num = random.randrange(0,max_num_task)
		while(new_num in random_num_array):
			new_num = random.randrange(0,max_num_task)
		random_num_array.append(new_num)
else:
	print(f"WARNING - All {max_num_task} tasks are used for finetuning")
	random_num_array = []
	for i in range(0,max_num_task):
		random_num_array.append(i)

#create training array and check array
indiv = [[] for Null in range(len(f_t_file_array))]

index_com = 0
for i in range(0,len(f_t_array)):
	tasks = f_t_array[i]
	for j in range(0,len(tasks)):
		if ((index_com+j)in random_num_array):
			indiv[0].append(tasks[j])
		else:
			indiv[1].append(tasks[j])
	index_com += len(f_t_array[i])

#create files
for i in range(0,len(indiv)):
	create_file_from_tasks(indiv[i],f_t_file_array[i])

In [10]:
#TRAIN fine tuning file
training_file_id = upload_file(f_t_file_array[0], "fine-tune")

job = client.fine_tuning.jobs.create(
		training_file = training_file_id,
		model = "gpt-4o-mini-2024-07-18",
	)

In [18]:
#CHECK fine tuning progress
f_t_job = client.fine_tuning.jobs.retrieve(job.id)
print(f_t_job)
print(f_t_job.status)
print(f_t_job.id)

FineTuningJob(id='ftjob-ZvXAyvZe9TbLEm5qwcWPmaSX', created_at=1746181496, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:ging-upm::BSiJ2Vo0', finished_at=1746183194, hyperparameters=Hyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-b9e6eTH4lj1kn4VhwdRfE1Rx', result_files=['file-GhURPw1wm5JsGs2QHMfTNm'], seed=1317882598, status='succeeded', trained_tokens=930231, training_file='file-GA81fiL71Z35VYxDBq2H1j', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3)), type='supervised'), user_provided_suffix=None, metadata=None, usage_metrics=None, shared_with_openai=True)
succeeded
ftjob-ZvXAyvZe9TbLEm5qwcWPmaSX


In [19]:
#EXTRACT fine tuned model
fine_tuned_model_id = f_t_job.fine_tuned_model
print(f_t_job)
print(fine_tuned_model_id)
with open("created_models.txt","a") as f:
	f.write(f"\"{f_t_job.id}\"\n")

FineTuningJob(id='ftjob-ZvXAyvZe9TbLEm5qwcWPmaSX', created_at=1746181496, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:ging-upm::BSiJ2Vo0', finished_at=1746183194, hyperparameters=Hyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-b9e6eTH4lj1kn4VhwdRfE1Rx', result_files=['file-GhURPw1wm5JsGs2QHMfTNm'], seed=1317882598, status='succeeded', trained_tokens=930231, training_file='file-GA81fiL71Z35VYxDBq2H1j', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3)), type='supervised'), user_provided_suffix=None, metadata=None, usage_metrics=None, shared_with_openai=True)
ft:gpt-4o-mini-2024-07-18:ging-upm::BSiJ2Vo0


In [10]:
def extract_input(new_line):
	return (new_line["body"]["messages"])

test_file = "middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_0.jsonl"
test_line = get_line_file(test_file,0,extract_input)

test_messages = []
test_messages.append(test_line[0])

response = client.chat.completions.create(
	model=fine_tuned_model_id, messages=test_messages, temperature=0
)

print(test_messages)
print(response.choices[0].message.content)

[{'role': 'user', 'content': '\nLa edad de adquisición (AoA) de una palabra se refiere a la edad en la que se aprendió una palabra por primera vez.\nEn concreto, cuándo una persona habría entendido por primera vez esa palabra si alguien la hubiera utilizado delante de ella, incluso cuando aún no la hubiera dicho, leído o escrito.\nEstima la edad media de adquisición (AoA) de la palabra "befabemí" para un hablante nativo de español.\nEl formato de salida debe ser un objeto JSON. Por ejemplo: { Word: befabemí , AoA: }\n'}]
{"Word": "befabem\u00ed", "AoA": "20.1989505089283"}


In [12]:
def generate_task(index,model,mess_content):
	task = {
        "custom_id": f"task-{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "temperature": 0,
            "response_format": { 
                "type": "json_object"
            },
            "messages": mess_content,
        }
    }
	return task

def create_task_from_json(json_object,index,prompt,model):
	word = json_object["Word"]
	desc = { "role": "user", "content": word_into_prompt(prompt,word) },
	task = generate_task(
		index,model,desc,
	)
	return task

def create_task_array_from_dataframe(df,prompt,model):
	tasks = []
	for index, row in df.iterrows():
		task = create_task_from_json(row,index,prompt,model)
		tasks.append(task)
	return tasks

def create_batch(file_name):
	batch_file = client.files.create(
		file = open(file_name, "rb"),
		purpose = "batch"
	)
	batch_job = client.batches.create(
		input_file_id = batch_file.id,
		endpoint = "/v1/chat/completions",
		completion_window = "24h"
	)
	return batch_job

def extract_input(new_line):
	return (new_line["messages"])

def extract_data(new_line):
	res = new_line["response"]["body"]["choices"][0]["message"]["content"]
	res = json.loads(res)
	return res

def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F\t]", "", text)
        return text.strip()
    return text


In [9]:
ft_job_id = "ftjob-U7DwTIa1h8Y3tAPA7J4d5vXq"
f_t_job = client.fine_tuning.jobs.retrieve(ft_job_id)
fine_tuned_model_id = f_t_job.fine_tuned_model
print(fine_tuned_model_id)

ft:gpt-4o-mini-2024-07-18:ging-upm::BQaA1KtD


In [ ]:
# DELETE
client.models.delete(fine_tuned_model_id)

In [22]:
#GENERATE BATCH FILE - from leftovers

test_file = f_t_file_array[1]
with open(test_file, 'r') as f:
	lines = len(f.readlines())

print("[lines]: "+str(lines))

batch_job_tasks = []
for i in range(0,lines):
	line = get_line_file(test_file,i,extract_input)
	task = generate_task(i,fine_tuned_model_id,[line[0]])
	batch_job_tasks.append(task)

create_file_from_tasks(batch_job_tasks,f_t_file_array[2])

[lines]: 5039


In [23]:
#GENERATE BATCH
ba_jo = create_batch(f_t_file_array[2])

In [31]:
#COMPLETION CHECK
batch = client.batches.retrieve(ba_jo.id)
result_file_id = batch.output_file_id
status = batch.status
print(batch)
print(status)

Batch(id='batch_6814a980cbac81908402857fb0dc6413', completion_window='24h', created_at=1746184576, endpoint='/v1/chat/completions', input_file_id='file-6pVT1uvn4UqkBc8qr9Y4wz', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1746185952, error_file_id=None, errors=None, expired_at=None, expires_at=1746270976, failed_at=None, finalizing_at=1746185391, in_progress_at=1746184579, metadata=None, output_file_id='file-F4XThq3V8PE8UMow6hCKNz', request_counts=BatchRequestCounts(completed=5039, failed=0, total=5039))
completed


In [32]:
#OUTPUT FILE
batch = client.batches.retrieve(batch.id)
result_file_id = batch.output_file_id

result = client.files.content(result_file_id).content

result_file_name = f_t_file_array[2].replace(".json","_result.json")
result_file_name = result_file_name.replace("middle_files","output_files")

with open(result_file_name, 'wb') as file:
	file.write(result)

In [33]:
out_file_1 = out_file("batch_job_mmlu_batch_aoa_alonso_second_result.jsonl")
middle_file_1 = middle_file("batch_job_mmlu_check_aoa_alonso_second.jsonl")
with open(out_file_1, 'r') as file:
	lines = len(file.readlines())

rows = []
errors = []

for i in range(0,lines):
	line = get_line_file(middle_file_1,i,extract_input)
	info = json.loads(line[1]["content"])
	initAoA = info["AoA"]
	word = info["Word"]
	AoA = ""
	try:
		AoA = get_line_file(out_file_1,i,extract_data)["AoA"]
	except:
		AoA = "NaN"

	try:
		if(AoA == "NaN"):
			mini_batch = [line[0]]
			response = client.chat.completions.create(
				model=fine_tuned_model_id, messages=mini_batch, temperature=0
			)
			out_obj = response.choices[0].message.content
			AoA = json.loads(out_obj)["AoA"]
	except:
		print(f"error in word: {word}")
		errors.append({"Line_Num":i, "word":word})

	rows.append({
		"word":word,
		"initAoA":initAoA,
		"AoA":AoA
	})

In [34]:
out_file_name = out_file("Results_f_t_corr_"+str(num_task_train)+"_second.xlsx")

clean_dtset,errors_dtset = pd.DataFrame(rows), pd.DataFrame(errors)

with pd.ExcelWriter(out_file_name) as writer:
	clean_dtset.to_excel(writer, sheet_name='Results',index=False)
	errors_dtset.to_excel(writer, sheet_name='Errors',index=False)

In [13]:
#GENERATE BATCH FILE - from dataset

dataset_folder = os.getenv("DATASET_FOLDER")
ddbb_name = "ProvResults_all_words.xlsx"
batch_ddbb_name = str(dataset_folder) + ddbb_name

df_analyze = pd.read_excel(batch_ddbb_name)
#df_analyze = df_analyze.sample(100)

#BASIC_MODEL = "gpt-4o-mini"
#f_t_array_analyze = create_task_array_from_dataframe(df_analyze,categorize_system_prompt_paraphrase,BASIC_MODEL)
f_t_array_analyze = create_task_array_from_dataframe(df_analyze,categorize_system_prompt_paraphrase,fine_tuned_model_id)

ddbb_batch_file = middle_file("batch_job_mmlu_batch_ddbb_aoa_alonso_second.jsonl")
create_file_from_tasks(f_t_array_analyze,ddbb_batch_file)

df_analyze.head()

,Word,NotFtAoA,AoA,Source
0,¡aba!,3.5,6.78,Hinojosa et al. (2021)
1,¡aba!,3.5,6.78,Hinojosa et al. (2021)
2,¡abur!,3.5,6.58,Hinojosa et al. (2021)
3,¡abur!,3.5,6.58,Hinojosa et al. (2021)
4,¡achís!,5.0,7.40,Hinojosa et al. (2021)


In [14]:
#DIVIDE TASK
def divide_task(tasks_array,file_array,task_index,num_tasks):
	res_task_array = []
	res_file_array = []
	task_array_to_div = []
	prov_file_name = ""
	for i in range(0,len(tasks_array)):
		if(i == task_index):
			task_array_to_div = tasks_array[i]
			prov_file_name = file_array[i]
		else:
			res_task_array.append(tasks_array[i])
			res_file_array.append(file_array[i])
	index = 1+int(len(task_array_to_div)/num_tasks)
	for i in range(0,index):
		tasks = []
		for j in range(0,num_tasks):
			if(i*num_tasks+j < len(task_array_to_div)):
				tasks.append(task_array_to_div[i*num_tasks+j])
		res_task_array.append(tasks)
		res_file_array.append(prov_file_name.replace(".json","_"+"0"*(1+int(index/10)-len(str(i)))+str(i)+".json"))
	return res_task_array,res_file_array

In [15]:
tasks_array = [f_t_array_analyze]
file_array = [ddbb_batch_file]

tasks_array,file_array = divide_task(tasks_array,file_array,0,10000)

#GENERATE TASK FILES
for i in range(0,len(tasks_array)):
	create_file_from_tasks(tasks_array[i],file_array[i])
	print(file_array[i])

middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_00.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_01.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_02.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_03.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_04.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_05.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_06.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_07.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_08.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_09.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_10.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_11.jsonl
middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_12.jsonl


In [16]:
#GENERATE BATCH
batch_jobs = []
for i in range(0,len(tasks_array)):
	ba_jo = create_batch(file_array[i])
	batch_jobs.append(ba_jo)

In [26]:
for i in range(0,len(batch_jobs)):
	batch = batch_jobs[i]
	batch = client.batches.retrieve(batch.id)
	print(batch)
	result_file_id = batch.output_file_id
	status = batch.status
	print(status)
	

Batch(id='batch_68274a9cef388190a3eb9e265abf4aae', completion_window='24h', created_at=1747405468, endpoint='/v1/chat/completions', input_file_id='file-Wkr5kZdVz5cagSVWduziDc', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1747410709, error_file_id=None, errors=None, expired_at=None, expires_at=1747491868, failed_at=None, finalizing_at=1747409854, in_progress_at=1747405471, metadata=None, output_file_id='file-BgrhdVcU91WLxiijvxBN9c', request_counts=BatchRequestCounts(completed=10000, failed=0, total=10000))
completed
Batch(id='batch_68274aa983c48190bf1e1992bf5419a5', completion_window='24h', created_at=1747405481, endpoint='/v1/chat/completions', input_file_id='file-93cC7bRv7DtVNdZQsy5vNq', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1747409255, error_file_id=None, errors=None, expired_at=None, expires_at=1747491881, failed_at=None, finalizing_at=1747408337, in_progress_at=1747405484, metadata=None, o

In [27]:
for i in range(0,len(batch_jobs)):
	batch = batch_jobs[i]
	batch = client.batches.retrieve(batch.id)
	if (batch.status == "completed"):

		result_file_id = batch.output_file_id

		result = client.files.content(result_file_id).content

		result_file_name = file_array[i].replace(".json","_result.json")

		with open(result_file_name, 'wb') as file:
			file.write(result)


In [28]:
#print(tasks_array)
print(file_array)

['middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_00.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_01.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_02.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_03.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_04.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_05.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_06.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_07.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_08.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_09.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_10.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_11.jsonl', 'middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_12.jsonl']


In [ ]:
def extract_in_file(line):
	line = line["body"]["messages"]
	return line

out_file_name = "output_files/New_Results_AoA_f_t_withoutft_second.xlsx"
#out_file_name = "output_files/New_Results_AoA_f_t_"+str(num_task_train)+"_second.xlsx"
file_name1 = "middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second.jsonl"
file_name2 = "middle_files/batch_job_mmlu_batch_ddbb_aoa_alonso_second_result.jsonl"

num_files = 13

rows = []
errors = []
acum = 0

for j in range(0, num_files):
	num = "0"*(1+int(num_files/10)-len(str(j)))+str(j)
	file_name1_new = file_name1.replace(".jsonl","_"+num+".jsonl")
	file_name2_new = file_name2.replace("result.jsonl",num+"_result.jsonl")
	with open(file_name2_new, 'r') as f:
		lines2 = len(f.readlines())

	for i in range(0,lines2):
	#for i in range(0,10):
		row = df_analyze.iloc[acum]
		try:
			line2 = get_line_file(file_name2_new,i,extract_data)
			#line2 = extract_data(lines2[i])
			newAoA = clean_text(line2["AoA"])
		except:
			newAoA = "NaN"
			#print(f"error in word: {row["Word"]}")
			#errors.append({"Line_Num":i, "file":j})
		
		try:
			if(newAoA == "NaN"):
				mini_batch = get_line_file(file_name1_new,i,extract_in_file)
				response = client.chat.completions.create(
					model=fine_tuned_model_id, messages=mini_batch, temperature=0
				)
				out_obj = response.choices[0].message.content
				newAoA = json.loads(out_obj)["AoA"]
		except:
			print(f"error in word: {row["Word"]}")
			errors.append({"Line_Num":i, "file":j})

		rows.append({
			"Word":row["Word"],
			"not_FT_AoA":row["NotFtAoA"],
			"first_FT_AoA":row["AoA"],
			"second_FT_AoA":newAoA,
			"source":row["Source"],
		})
		acum += 1

clean_dtset,errors_dtset = pd.DataFrame(rows), pd.DataFrame(errors)

with pd.ExcelWriter(out_file_name) as writer:
	clean_dtset.to_excel(writer, sheet_name='Results',index=False)
	errors_dtset.to_excel(writer, sheet_name='Errors',index=False)

error in word: ¡adiós!
error in word: águila imperial
error in word: áliger
error in word: allemande


In [ ]:
#SAVE ouput in a .xlsx file
file_name = out_file("New_Results_AoA_f_t_2000_second.xlsx")
clean_dtset = pd.DataFrame(rows)

#CALCULATE correlation coeff

cmp_clm_1 = "not_FT_AoA"
cmp_clm_2 = "FT_AoA"

pearson_corr = clean_dtset[[cmp_clm_1, cmp_clm_2]].corr('pearson')
pearson_corr = pearson_corr[cmp_clm_1][cmp_clm_2]
print(f"Pearson_Corr: {pearson_corr}")

spearman_corr = clean_dtset[[cmp_clm_1, cmp_clm_2]].corr('spearman')
spearman_corr = spearman_corr[cmp_clm_1][cmp_clm_2]
print(f"Spearman_Corr: {spearman_corr}")
